In [1]:
import numpy as np
from dataset_generator.data_generator import *
from utils.other_utils import *
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
from model.CTRNN import *
from easydict import EasyDict as edict
import yaml
from utils.train_helper import load_model
from sklearn.decomposition import PCA
import plotly
import plotly.graph_objs as go
import time

# Creating data

In [2]:
# data
data_duration = 70
dt  = 0.01
f_beat_time = 4
interval = 0.8
n_of_targets = 10
n_of_beats = 10
target_shape = 'full_sine'
data_x, data_y, first_beat_idx, _, _ = beats_chain_forced_generator(number_of_beats=n_of_beats,
                                                                    number_of_targets=n_of_targets,
                                                                    target_starts_from_beat=2,
                                                                    target_shape=target_shape,
                                                                    data_duration=data_duration,
                                                                    first_beat_time=f_beat_time,
                                                                    interval=interval,
                                                                    target_amplitude=1,
                                                                    beats_amplitude=1,
                                                                    dt=dt)

# model

In [3]:
# model
from model.CTRNN import *
from easydict import EasyDict as edict
import yaml
from utils.train_helper import load_model

config_file_path = '/home/matin/McMaster/trimba/phd_codes/CTRNN/experiments/ctrnn_v2/beats_chain/CTRNN_beats_chain_full_sine_beatsn10_targetsn10_dt0.01_2023-Feb-13-00-20-33_2965118/config.yaml'
config = edict(yaml.full_load(open(config_file_path, 'r')))
config.device = 'cpu'
model = CTRNN(config)
model_file = os.path.join(config.save_dir, config.test.test_model_name)
model_file = model_file.replace('/home/mtnusf97/projects/def-cannoj9/mtnusf97/CTRNN/', '../')
load_model(model, model_file, config.device)
model.to(config.device)

CTRNN(
  (wI): Linear(in_features=1, out_features=500, bias=True)
  (wR): Linear(in_features=500, out_features=500, bias=False)
  (wO): Linear(in_features=500, out_features=1, bias=True)
)

# predict

In [4]:
activations, hidden_states, outputs = model.init_activations_outputs(batch_size=1)
outputs, all_activations, all_hidden_states = model.detailed_predict(data_x,
                                                             activations.to(config.device),
                                                             hidden_states.to(config.device),
                                                             outputs)

all_activations = np.array([i.flatten().detach().numpy() for i in all_activations])
all_hidden_states = np.array([i.flatten().detach().numpy() for i in all_hidden_states])

# PCA

In [5]:
pca = PCA(n_components=3)
pca.fit(all_activations)
print(pca.explained_variance_ratio_)
print(np.sum(pca.explained_variance_ratio_))

[0.5071629  0.26734036 0.10595677]
0.8804601


In [6]:
activations_reduced = pca.transform(all_activations)

In [7]:
len(activations_reduced[:,0])

7000

# Plot

In [ ]:
# Configure Plotly to be rendered inline in the notebook.
plotly.offline.init_notebook_mode()

# Configure the trace.
trace = go.Scatter3d(
    x=activations_reduced[:,0],  # <-- Put your data instead
    y=activations_reduced[:,1],  # <-- Put your data instead
    z=activations_reduced[:,2],  # <-- Put your data instead
    mode='markers',
    marker={
        'size': 2,
        'opacity': 1,
        'color': 'red'
    }
)

# Configure the layout.
layout = go.Layout(
    margin={'l': 0, 'r': 0, 'b': 0, 't': 0}
)

data = [trace]

plot_figure = go.Figure(data=data, layout=layout)
# plot_figure = go.Figure(data=data)

# Render the plot.
# plotly.offline.iplot(plot_figure)
plot_figure.show()



# try animation

In [41]:
# Create figure
x=activations_reduced[:,0][:100]
y=activations_reduced[:,1][:100]
z=activations_reduced[:,2][:100]

In [42]:
data_0 = go.Scatter3d(x=[], y=[], z=[],
                      mode='markers',
                      marker={'size': 2,'opacity': 1,'color': 'red'})

In [43]:
layout = go.Layout(
    margin={'l': 0, 'r': 0, 'b': 0, 't': 0}
)

fig = go.Figure(data=[data_0], layout=layout)
#
# fig.update_layout(scene = dict(
#         xaxis=dict(range=[min(x), max(x)], autorange=False),
#         yaxis=dict(range=[min(y), max(y)], autorange=False),
#         zaxis=dict(range=[min(z), max(z)], autorange=False),
#         ))

In [45]:
frames = [go.Frame(data= [go.Scatter3d(
                                       x=x[:k+1],
                                       y=y[:k+1],
                                       z=z[:k+1])],

                   traces= [0],
                   name=f'frame{k}'
                  )for k  in  range(len(x)-1)]

In [ ]:
fig.update(frames=frames)
fig.update_layout(updatemenus=[dict(type="buttons",
                          buttons=[dict(label="Play",
                                        method="animate",
                                        args=[None, dict(frame=dict(redraw=True,fromcurrent=True, mode='immediate'))      ])])])

plotly.offline.iplot(fig)
# fig.show()

# test

In [58]:
import numpy as np
import plotly.graph_objects as go

# x,y,z = np.genfromtxt(r'dat.txt', unpack=True)

In [62]:
# x = np.arange(30)
# y = np.arange(30) + 2
# z = np.arange(30) - 2

x = activations_reduced[:,0][:1000]
y = activations_reduced[:,1][:1000]
z = activations_reduced[:,2][:1000]

In [ ]:
# Create figure
fig = go.Figure(go.Scatter3d(x=[], y=[], z=[],
                             mode="markers",
                             marker=dict(color="red", size=2)
                             )
                )


# Frames
frames = [go.Frame(data= [go.Scatter3d(x=x[:k+1],
                                       y=y[:k+1],
                                       z=z[:k+1]
                                       )
                          ],
                   traces= [0],
                   name=f'frame{k}'
                  )for k  in  range(len(x)-1)
          ]

fig.update(frames=frames)




def frame_args(duration):
    return {
            "frame": {"duration": duration},
            "mode": "immediate",
            "fromcurrent": True,
            "transition": {"duration": duration, "easing": "linear"},
            }


sliders = [
    {"pad": {"b": 10, "t": 60},
     "len": 0.9,
     "x": 0.1,
     "y": 0,

     "steps": [
                 {"args": [[f.name], frame_args(0)],
                  "label": str(k),
                  "method": "animate",
                  } for k, f in enumerate(fig.frames)
              ]
     }
        ]

fig.update_layout(

    updatemenus = [{"buttons":[
                    {
                        "args": [None, frame_args(5)],
                        "label": "Play",
                        "method": "animate",
                    },
                    {
                        "args": [[None], frame_args(0)],
                        "label": "Pause",
                        "method": "animate",
                  }],
                "font": {"color":"red"},
                "direction": "left",
                "pad": {"r": 10, "t": 70},
                "type": "buttons",
                "x": 0.1,
                "y": 0,
            }
         ],
         sliders=sliders
    )

fig.update_layout(scene = dict(xaxis=dict(range=[min(x), max(x)], autorange=False),
                               yaxis=dict(range=[min(y), max(y)], autorange=False),
                               zaxis=dict(range=[min(z), max(z)], autorange=False)
                               )
                  )

fig.update_layout(sliders=sliders)
fig.show()